In [10]:
import pandas as pd
import numpy as np
import re

train = pd.read_csv("train_data.csv")
test = pd.read_csv("test_data.csv")


def clean(x):
    return re.sub(r"[^\w]", "", str(x).lower())

train["word_norm"] = train["word"].apply(clean)
test["word_norm"] = test["word"].apply(clean)

def parse(x):
    s = x.split("_")
    return int(s[-1]), int(s[-2]) if s[-2].isdigit() else 1

train["word_pos"], train["page"] = zip(*train["word_id"].apply(parse))
test["word_pos"], test["page"] = zip(*test["word_id"].apply(parse))


nz = train[train["answer"] > 0]

word_mean = nz.groupby("word_norm")["answer"].mean()
word_freq = train["word_norm"].value_counts()

total = train.groupby("word_norm")["answer"].count()
zero = train.assign(z=(train["answer"]==0)).groupby("word_norm")["z"].sum()

word_skip = zero / total

gmean = nz["answer"].mean()
gskip = (train["answer"]==0).mean()

ps = nz.groupby("participant_id")["answer"].mean() / gmean


all_text = pd.concat([train["word"], test["word"]])
chars = sorted(set("".join(all_text)))

char2idx = {"PAD":0,"UNK":1}
for i,c in enumerate(chars):
    char2idx[c] = i+2

def to_ids(w):
    ids = [char2idx.get(c,1) for c in str(w)[:30]]
    ids += [0]*(30-len(ids))
    return ids

train["char_ids"] = train["word"].apply(to_ids)
test["char_ids"] = test["word"].apply(to_ids)


def feats(df):
    w = df["word_norm"]

    f = np.column_stack([
        w.map(word_mean).fillna(gmean).values / 1000,
        w.map(word_skip).fillna(gskip).values,
        w.map(word_freq).fillna(0).values / 100,
        df["word_pos"].values / 100
    ]).astype(np.float32)

    return f

train_feats = feats(train)
test_feats = feats(test)


def make(df, feats, y=True):
    out = []
    df = df.copy()
    df["i"] = np.arange(len(df))

    for (t,p,pg), g in df.groupby(["text","participant_id","page"]):
        g = g.sort_values("word_pos")

        idx = g["i"].values

        out.append({
            "c": g["char_ids"].tolist(),
            "f": feats[idx],
            "y": g["answer"].values if y else None,
            "s": ps.get(p,1.0),
            "len": len(g),
            "id": g.get("datapointID", pd.Series([None]*len(g))).values,
            "meta": list(zip(g["text"], g["participant_id"], g["page"], g["word_pos"]))
        })

    return out

train_seqs = make(train, train_feats, True)
test_seqs = make(test, test_feats, False)


import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

class M(nn.Module):
    def __init__(self):
        super().__init__()

        self.e = nn.Embedding(len(char2idx),32)
        self.c = nn.Conv1d(32,64,3,padding=1)

        self.l = nn.LSTM(64+4,128,batch_first=True,bidirectional=True)

        self.sp = nn.Sequential(
            nn.Linear(1,32),
            nn.Tanh(),
            nn.Linear(32,256),
            nn.Sigmoid()
        )

        self.fc = nn.Sequential(
            nn.LayerNorm(256),
            nn.Linear(256,64),
            nn.GELU()
        )

        self.g = nn.Linear(64,1)
        self.r = nn.Linear(64,1)

    def forward(self,x,f,s,lens):
        B,T,C = x.shape

        x = x.view(B*T,C)
        x = self.e(x)
        x = x.permute(0,2,1)
        x = F.gelu(self.c(x))
        x = x.max(2).values
        x = x.view(B,T,64)

        x = torch.cat([x,f],-1)

        x = nn.utils.rnn.pack_padded_sequence(x,lens,batch_first=True,enforce_sorted=False)
        x,_ = self.l(x)
        x,_ = nn.utils.rnn.pad_packed_sequence(x,batch_first=True)

        s = self.sp(s.unsqueeze(1)).unsqueeze(1)
        x = x * s

        h = self.fc(x)

        g = torch.sigmoid(self.g(h)).squeeze(-1)
        t = F.softplus(self.r(h)).squeeze(-1)*200+100

        return g,t

model = M().to(device)


def batch(b):
    import torch

    m = max(x["len"] for x in b)

    X,F,Y,S,M,L = [],[],[],[],[],[]

    for x in b:
        l = x["len"]
        L.append(l)

        X.append(x["c"] + [[0]*30]*(m-l))
        F.append(np.vstack([x["f"], np.zeros((m-l,4))]))

        S.append(x["s"])

        if x["y"] is not None:
            Y.append(list(x["y"]) + [0]*(m-l))

        M.append([1]*l + [0]*(m-l))

    X = torch.tensor(X)
    F = torch.tensor(F).float()
    M = torch.tensor(M).bool()
    L = torch.tensor(L)
    S = torch.tensor(S).float()

    if len(Y):
        Y = torch.tensor(Y).float()
    else:
        Y = None

    return X,F,Y,S,M,L


def loss(g,t,y,m):
    gt = (y>0).float()

    b = F.binary_cross_entropy(g[m], gt[m])

    nz = m & (y>0)

    if nz.sum()==0:
        return b

    p = t[nz]/1000
    r = y[nz]/1000

    l = F.l1_loss(p,r)

    return l + 0.3*b


from torch.utils.data import DataLoader

loader = DataLoader(train_seqs, batch_size=32, shuffle=True, collate_fn=batch)

import torch.optim as optim

opt = optim.AdamW(model.parameters(), lr=3e-3)

for e in range(5):
    model.train()
    s=0

    for X,F,Y,S,M,L in loader:
        X,F,Y,S,M = X.to(device),F.to(device),Y.to(device),S.to(device),M.to(device)

        opt.zero_grad()
        g,t = model(X,F,S,L)
        l = loss(g,t,Y,M)
        l.backward()
        opt.step()

        s+=l.item()

    print(e, s/len(loader))


def predict(seqs):
    model.eval()
    out_g,out_t = [],[]

    with torch.no_grad():
        for x in seqs:
            X = torch.tensor([x["c"]]).to(device)
            F = torch.tensor([x["f"]]).float().to(device)
            S = torch.tensor([x["s"]]).float().to(device)
            L = torch.tensor([x["len"]])

            g,t = model(X,F,S,L)

            out_g.append(g[0][:x["len"]].cpu().numpy())
            out_t.append(t[0][:x["len"]].cpu().numpy())

    return np.concatenate(out_g), np.concatenate(out_t)


g,t = predict(train_seqs)

y = train["answer"].values

best_t, best_s = 0.5, 0

from sklearn.metrics import r2_score
from scipy.stats import pearsonr

for th in np.arange(0.2,0.8,0.05):
    p = np.where(g<th,0,t)

    r2 = max(0,r2_score(y,p))
    pr = abs(pearsonr(y,p)[0])

    s = (r2+pr)/2

    if s>best_s:
        best_s = s
        best_t = th


g,t = predict(test_seqs)

preds = np.where(g<best_t,0,t)

sub = pd.DataFrame({
    "subtaskID":1,
    "datapointID":test["datapointID"],
    "answer":preds
})

sub.to_csv("submission_lstm.csv", index=False)

AttributeError: 'Tensor' object has no attribute 'GELU'